In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

from langgraph.graph import StateGraph, END, START
from typing import TypedDict, List
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM 초기화
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# Pinecone 연결
embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = PineconeVectorStore.from_existing_index(
    index_name="finance-bok",
    embedding=embeddings,
    namespace="bok-ns1"
)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "namespace": "bok-ns1"}
)

print("초기화 완료")

c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


초기화 완료


In [3]:
class FinanceRAGState(TypedDict):
    query:           str           # 사용자 원본 질문
    rewritten_query: str           # 재작성된 검색 질문
    docs:            List[Document] # 검색된 문서
    answer:          str           # 최종 답변
    retry_count:     int           # 재시도 횟수 (최대 2회)
    doc_grade:       str           # 문서 품질 평가 결과 ("good" / "bad")

In [4]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 금융 검색 전문가입니다.
사용자의 질문을 한국은행 경제전망 보고서 검색에 최적화된 형태로 재작성하세요.
재작성된 질문만 출력하세요."""),
    ("human", "{query}")
])
rewrite_chain = rewrite_prompt | llm | parser

def rewrite_node(state: FinanceRAGState):
    rewritten = rewrite_chain.invoke({"query": state["query"]})
    print(f"[rewrite] {rewritten}")
    return {
        "rewritten_query": rewritten,
        "retry_count": state.get("retry_count", 0)
    }

In [5]:
def retrieve_node(state: FinanceRAGState):
    docs = retriever.invoke(state["rewritten_query"])
    print(f"[retrieve] {len(docs)}개 문서 검색")
    return {"docs": docs}

In [6]:
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 검색 문서의 품질을 평가하는 전문가입니다.
아래 질문과 검색된 문서를 보고 문서가 질문에 답변하기에 충분한지 평가하세요.
충분하면 "good", 불충분하면 "bad"만 출력하세요."""),
    ("human", """질문: {query}

검색된 문서:
{context}""")
])
grade_chain = grade_prompt | llm | parser

def grade_node(state: FinanceRAGState):
    context = "\n\n".join([doc.page_content for doc in state["docs"]])
    grade   = grade_chain.invoke({
        "query":   state["query"],
        "context": context
    }).strip().lower()
    print(f"[grade] 품질 평가 결과: {grade} (재시도 횟수: {state.get('retry_count', 0)})")
    return {"doc_grade": grade}

In [7]:
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 한국은행 경제전망 보고서 전문 애널리스트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고 문서]
{context}"""),
    ("human", "{question}")
])
generate_chain = generate_prompt | llm | parser

def generate_node(state: FinanceRAGState):
    context = "\n\n".join([doc.page_content for doc in state["docs"]])
    answer  = generate_chain.invoke({
        "context":  context,
        "question": state["query"]
    })
    print(f"[generate] 답변 생성 완료")
    return {"answer": answer}

In [8]:
def fallback_node(state: FinanceRAGState):
    print(f"[fallback] 재시도 초과 → 일반 답변 생성")
    answer = generate_chain.invoke({
        "context":  "\n\n".join([doc.page_content for doc in state["docs"]]),
        "question": state["query"]
    })
    return {"answer": f"[참고: 검색 품질이 낮을 수 있습니다]\n{answer}"}

In [9]:
def rewrite_retry_node(state: FinanceRAGState):
    retry     = state.get("retry_count", 0) + 1
    rewritten = rewrite_chain.invoke({"query": state["query"]})
    print(f"[rewrite_retry] 재시도 {retry}회 → {rewritten}")
    return {
        "rewritten_query": rewritten,
        "retry_count":     retry
    }

In [10]:
def route_after_grade(state: FinanceRAGState) -> str:
    if state["doc_grade"] == "good":
        return "generate"

    retry = state.get("retry_count", 0)
    if retry >= 2:
        return "fallback"

    return "rewrite_retry"

In [11]:
graph = StateGraph(FinanceRAGState)

# Node 등록
graph.add_node("rewrite",       rewrite_node)
graph.add_node("retrieve",      retrieve_node)
graph.add_node("grade",         grade_node)
graph.add_node("generate",      generate_node)
graph.add_node("fallback",      fallback_node)
graph.add_node("rewrite_retry", rewrite_retry_node)

# Edge 연결
graph.add_edge(START,      "rewrite")
graph.add_edge("rewrite",  "retrieve")
graph.add_edge("retrieve", "grade")

# 조건 분기 — grade 이후
graph.add_conditional_edges(
    "grade",
    route_after_grade,
    {
        "generate":      "generate",
        "rewrite_retry": "rewrite_retry",
        "fallback":      "fallback"
    }
)

# 재시도 루프
graph.add_edge("rewrite_retry", "retrieve")

# 종료
graph.add_edge("generate", END)
graph.add_edge("fallback",  END)

app = graph.compile()
print("Graph 컴파일 성공")

Graph 컴파일 성공


In [12]:
initial_state = {
    "query":       "2026년 GDP 성장률 전망치는 얼마인가요?",
    "retry_count": 0
}
final = app.invoke(initial_state)
print(f"\n[ 최종 답변 ]\n{final['answer']}")

[rewrite] 2026년 한국은행 경제전망 보고서에 따른 GDP 성장률 전망치는 얼마인가요?
[retrieve] 3개 문서 검색
[grade] 품질 평가 결과: good (재시도 횟수: 0)
[generate] 답변 생성 완료

[ 최종 답변 ]
2026년 GDP 성장률 전망치는 2.0%입니다.
